In [64]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

0

# Local Ensured Plotly

This notebook demonstrates the canonical Plotly ensured style `plotly.local.ensured`. The older style name `plotly.local.ensured_triangular` is deprecated.

Ensured plot semantics remain aligned with CE's native view:

- x-axis = probability in probabilistic mode, prediction in regression mode
- y-axis = uncertainty
- original marker = original prediction
- arrows = predictive movement from the original point to alternative or rule points
- hover reveals rule conditions and interval metadata
- feature controls provide searchable hide/show toggles for feature groups
- side panel is a text detail view that updates when a rule point is clicked
- roles such as counterfactual, counterpotential, semifactual, ensured, and pareto are shown only when available or defensibly inferable
- arrows and alternatives do not imply causal actionability
- `filter_top` keeps dense ensured plots readable

In [66]:
import importlib
import sys
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
import calibrated_explanations.plugins.registry as registry
import ce_visualization_plotly.ensured as ensured_module
import ce_visualization_plotly.plugin as plotly_plugin_module
from sklearn.datasets import make_classification, make_regression
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

reset_catalog = getattr(registry, 'reset_plugin_catalog', None)
if callable(reset_catalog):
    reset_catalog(kind='all')
clear_env_cache = getattr(registry, 'clear_env_trust_cache', None)
if callable(clear_env_cache):
    clear_env_cache()
clear_warnings = getattr(registry, 'clear_trust_warnings', None)
if callable(clear_warnings):
    clear_warnings()
for module_name in [name for name in list(sys.modules) if name.startswith('ce_visualization_plotly')]:
    sys.modules.pop(module_name, None)
import ce_visualization_plotly.ensured as ensured_module
import ce_visualization_plotly.plugin as plotly_plugin_module
importlib.reload(ensured_module)
importlib.reload(plotly_plugin_module)
plotly_plugin_module.register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

## Classification Example

The current CE alternative custom-style routing works at the collection level, so the ensured examples below use `explore_alternatives(...)` and call `.plot(...)` on the returned collection with a single query instance.

When `feature_checklist=True`, the plot renders a searchable feature control panel. When `side_panel=True`, click a blue rule point to populate the text detail panel.

In [67]:
X_cls, y_cls = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=7,
)

x_proper_cls, x_holdout_cls, y_proper_cls, y_holdout_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.40,
    random_state=7,
    stratify=y_cls,
)
x_cal_cls, X_query_cls, y_cal_cls, y_query_cls = train_test_split(
    x_holdout_cls,
    y_holdout_cls,
    test_size=0.50,
    random_state=7,
    stratify=y_holdout_cls,
)

classification_model = LogisticRegression(random_state=7, solver='liblinear')
classification_explainer = WrapCalibratedExplainer(classification_model)
classification_explainer.fit(x_proper_cls, y_proper_cls)
assert classification_explainer.fitted is True
classification_explainer.calibrate(x_cal_cls, y_cal_cls)
assert classification_explainer.calibrated is True
classification_alternatives = classification_explainer.explore_alternatives(X_query_cls[:1])
len(classification_alternatives.explanations)

1

In [68]:
classification_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
)
classification_plot

PlotRenderResult(artifact={'artifact_type': 'plotly.local.ensured', 'artifact_version': '0.2.0', 'style': 'plotly.local.ensured', 'base_plotspec_kind': 'triangular', 'mode': 'classification', 'task': 'classification', 'original': {'id': 'original-point', 'kind': 'original', 'prediction': 0.8461538461538463, 'uncertainty': 0.18181818181818177, 'low': 0.8181818181818182, 'high': 1.0, 'label': 'Original Prediction', 'hover': 'Original prediction<br>Prediction: 0.846154<br>Uncertainty: 0.181818<br>Interval: [0.818182, 1]'}, 'rule_points': [{'id': 'rule-point-1', 'kind': 'rule', 'index': 1, 'feature_index': 0, 'feature_name': '0', 'true_value': -0.6213494404078027, 'instance_value': -0.6213494404078027, 'rule': '0 > 2.52', 'alternative_value': [2.9443172518000713, 3.243391746818367, 3.7286380013905953], 'prediction': 0.8993265993265993, 'uncertainty': 0.11203703703703705, 'low': 0.887962962962963, 'high': 1.0, 'delta_prediction': 0.05317275317275305, 'delta_uncertainty': -0.0697811447811447

In [69]:
classification_checklist_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
    feature_checklist=True,
)
classification_checklist_plot

PlotRenderResult(artifact={'artifact_type': 'plotly.local.ensured', 'artifact_version': '0.2.0', 'style': 'plotly.local.ensured', 'base_plotspec_kind': 'triangular', 'mode': 'classification', 'task': 'classification', 'original': {'id': 'original-point', 'kind': 'original', 'prediction': 0.8461538461538463, 'uncertainty': 0.18181818181818177, 'low': 0.8181818181818182, 'high': 1.0, 'label': 'Original Prediction', 'hover': 'Original prediction<br>Prediction: 0.846154<br>Uncertainty: 0.181818<br>Interval: [0.818182, 1]'}, 'rule_points': [{'id': 'rule-point-1', 'kind': 'rule', 'index': 1, 'feature_index': 0, 'feature_name': '0', 'true_value': -0.6213494404078027, 'instance_value': -0.6213494404078027, 'rule': '0 > 2.52', 'alternative_value': [2.9443172518000713, 3.243391746818367, 3.7286380013905953], 'prediction': 0.8993265993265993, 'uncertainty': 0.11203703703703705, 'low': 0.887962962962963, 'high': 1.0, 'delta_prediction': 0.05317275317275305, 'delta_uncertainty': -0.0697811447811447

In [77]:
classification_panel_plot = classification_alternatives.add_conjunctions().plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
    side_panel=True,
)
# classification_panel_plot

In [78]:
classification_full_plot = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)

In [72]:
classification_export = classification_alternatives.plot(
    style='plotly.local.ensured',
    show=False,
    filename='ensured_classification.html',
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)
classification_export.saved_paths

('ensured_classification.html',)

## Regression Example

Regression ensured plots use the same Plotly style, but the x-axis represents the calibrated prediction value instead of a class probability.

In [73]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=8,
    n_informative=5,
    noise=0.2,
    random_state=11,
)

x_proper_reg, x_holdout_reg, y_proper_reg, y_holdout_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.40,
    random_state=11,
)
x_cal_reg, X_query_reg, y_cal_reg, y_query_reg = train_test_split(
    x_holdout_reg,
    y_holdout_reg,
    test_size=0.50,
    random_state=11,
)

regression_model = LinearRegression()
regression_explainer = WrapCalibratedExplainer(regression_model)
regression_explainer.fit(x_proper_reg, y_proper_reg)
assert regression_explainer.fitted is True
regression_explainer.calibrate(x_cal_reg, y_cal_reg)
assert regression_explainer.calibrated is True
regression_alternatives = regression_explainer.explore_alternatives(
    X_query_reg[:1],
    low_high_percentiles=(10, 90),
)
len(regression_alternatives.explanations)

1

In [74]:
regression_plot = regression_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
)
regression_plot

PlotRenderResult(artifact={'artifact_type': 'plotly.local.ensured', 'artifact_version': '0.2.0', 'style': 'plotly.local.ensured', 'base_plotspec_kind': 'triangular', 'mode': 'regression', 'task': 'regression', 'original': {'id': 'original-point', 'kind': 'original', 'prediction': 71.63480375265759, 'uncertainty': 0.48670903777272656, 'low': 71.4194818488862, 'high': 71.90619088665893, 'label': 'Original Prediction', 'hover': 'Original prediction<br>Prediction: 71.6348<br>Uncertainty: 0.486709<br>Interval: [71.4195, 71.9062]'}, 'rule_points': [{'id': 'rule-point-5', 'kind': 'rule', 'index': 5, 'feature_index': 2, 'feature_name': '2', 'true_value': -1.0054241520424556, 'instance_value': -1.0054241520424556, 'rule': '2 > -0.66', 'alternative_value': [-0.2848362024953862, 0.22547109017094458, 0.8230289908428537], 'prediction': 145.5744023488251, 'uncertainty': 0.486709037772755, 'low': 145.3590804450537, 'high': 145.84578948282646, 'delta_prediction': 73.9395985961675, 'delta_uncertainty':

In [75]:
regression_full_plot = regression_alternatives.plot(
    style='plotly.local.ensured',
    show=True,
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)
regression_full_plot

PlotRenderResult(artifact={'artifact_type': 'plotly.local.ensured', 'artifact_version': '0.2.0', 'style': 'plotly.local.ensured', 'base_plotspec_kind': 'triangular', 'mode': 'regression', 'task': 'regression', 'original': {'id': 'original-point', 'kind': 'original', 'prediction': 71.63480375265759, 'uncertainty': 0.48670903777272656, 'low': 71.4194818488862, 'high': 71.90619088665893, 'label': 'Original Prediction', 'hover': 'Original prediction<br>Prediction: 71.6348<br>Uncertainty: 0.486709<br>Interval: [71.4195, 71.9062]'}, 'rule_points': [{'id': 'rule-point-5', 'kind': 'rule', 'index': 5, 'feature_index': 2, 'feature_name': '2', 'true_value': -1.0054241520424556, 'instance_value': -1.0054241520424556, 'rule': '2 > -0.66', 'alternative_value': [-0.2848362024953862, 0.22547109017094458, 0.8230289908428537], 'prediction': 145.5744023488251, 'uncertainty': 0.486709037772755, 'low': 145.3590804450537, 'high': 145.84578948282646, 'delta_prediction': 73.9395985961675, 'delta_uncertainty':

In [76]:
regression_export = regression_alternatives.plot(
    style='plotly.local.ensured',
    show=False,
    filename='ensured_regression.html',
    filter_top=20,
    feature_checklist=True,
    side_panel=True,
)
regression_export.saved_paths

('ensured_regression.html',)